
#  Semana 02 — Ejercicios de Optimización con PuLP

## Cuaderno de trabajo para estudiantes

Este notebook contiene **únicamente los planteamientos de los ejercicios**.  
El objetivo es que cada estudiante formule y programe su propia solución utilizando **PuLP en Python**.

---

##  Indicaciones generales

Para cada ejercicio se recomienda seguir esta secuencia:

1. Identificar las **variables de decisión**.
2. Determinar si son **continuas, enteras o binarias**.
3. Formular la **función objetivo**.
4. Escribir matemáticamente las **restricciones**.
5. Implementar el modelo en **PuLP**.
6. Resolver el modelo.
7. Revisar el estado de la solución.
8. Validar manualmente las restricciones.
9. Interpretar el resultado en el contexto del problema.

> 💡 No basta con obtener números. Debe justificarse por qué la solución encontrada es factible y qué significa en el contexto del problema.



##  Preparación del entorno

Utilice la siguiente celda únicamente para importar PuLP.

Si la librería no está instalada en su entorno, instálela antes de continuar.


In [2]:

# Importar PuLP
import pulp



#  Ejercicio 1 — Dimensionamiento de infraestructura Cloud

##  Planteamiento

Una empresa debe contratar instancias de tres tipos para soportar una nueva plataforma.  
Se desea cubrir una capacidad mínima de **CPU** y **memoria RAM** al menor costo mensual posible.

### 📊 Datos

| Tipo | Costo mensual | vCPU | RAM |
|---|---:|---:|---:|
| A — Standard | $120 | 8 | 32 GB |
| B — Compute | $180 | 16 | 64 GB |
| C — High Capacity | $260 | 32 | 96 GB |

###  Condiciones

- Se requieren al menos **160 vCPU**.
- Se requieren al menos **520 GB de RAM**.
- Por resiliencia, deben contratarse al menos **3 instancias tipo C**.
- No pueden administrarse más de **15 instancias en total**.

---

##  Trabajo del estudiante

Formule y resuelva el modelo en PuLP.

Debe identificar:

- variables de decisión;
- tipo de variables;
- función objetivo;
- restricciones;
- solución óptima;
- costo mínimo;
- validación de CPU, RAM y número total de instancias;
- interpretación de la solución.


In [3]:
import pulp

problema = pulp.LpProblem("Infraestructura_Cloud", pulp.LpMinimize)

# Variables
A = pulp.LpVariable("A_Standard", lowBound=0, cat="Integer")
B = pulp.LpVariable("B_Compute", lowBound=0, cat="Integer")
C = pulp.LpVariable("C_HighCapacity", lowBound=0, cat="Integer")

# Funcion objetivo
problema += 120*A + 180*B + 260*C

# Restricciones
problema += 8*A + 16*B + 32*C >= 160
problema += 32*A + 64*B + 96*C >= 520
problema += C >= 3
problema += A + B + C <= 15

# Resolver
problema.solve()

print("Estado:", pulp.LpStatus[problema.status])
print("A =", A.value())
print("B =", B.value())
print("C =", C.value())
print("Costo minimo =", pulp.value(problema.objective))

print("CPU =", 8*A.value() + 16*B.value() + 32*C.value())
print("RAM =", 32*A.value() + 64*B.value() + 96*C.value())
print("Total =", A.value() + B.value() + C.value())

Estado: Optimal
A = 0.0
B = 1.0
C = 5.0
Costo minimo = 1480.0
CPU = 176.0
RAM = 544.0
Total = 6.0



##  Reto de ampliación

Modifique el modelo anterior considerando ahora:

- demanda mínima de **200 vCPU**;
- demanda mínima de **640 GB de RAM**;
- obligación de contratar al menos **2 instancias tipo A** por compatibilidad con software legado.

Compare el nuevo costo con el modelo original.


In [4]:
import pulp

# 1. Definición del problema
problema = pulp.LpProblem("Cloud_Reto", pulp.LpMinimize)

# 2. Variables de decisión
A = pulp.LpVariable("A_Standard", lowBound=0, cat="Integer")
B = pulp.LpVariable("B_Compute", lowBound=0, cat="Integer")
C = pulp.LpVariable("C_HighCapacity", lowBound=0, cat="Integer")

# 3. Función objetivo
problema += 120 * A + 180 * B + 260 * C, "Costo_Total"

# 4. Restricciones
problema += 8 * A + 16 * B + 32 * C >= 200, "Min_CPU"
problema += 32 * A + 64 * B + 96 * C >= 640, "Min_RAM"
problema += A >= 2, "Min_A"
problema += A + B + C <= 15, "Max_Instancias"

# 5. Resolver
problema.solve()

# 6. Salidas (usando .varValue)
print("Estado:", pulp.LpStatus[problema.status])

a_val = int(A.varValue) if A.varValue is not None else 0
b_val = int(B.varValue) if B.varValue is not None else 0
c_val = int(C.varValue) if C.varValue is not None else 0

print(f"A = {a_val}")
print(f"B = {b_val}")
print(f"C = {c_val}")
print(f"Costo = ${pulp.value(problema.objective):.2f}")


Estado: Optimal
A = 2
B = 0
C = 6
Costo = $1800.00



#  Ejercicio 2 — Enrutamiento de tráfico entre enlaces WAN

##  Planteamiento

Un centro de datos debe distribuir **1,000 Mbps** entre tres enlaces WAN.  
Los enlaces tienen distintos costos, capacidades y latencias.

Se desea **minimizar el costo del tráfico**, pero la latencia promedio ponderada no debe superar **40 ms**.

###  Datos

| Enlace | Costo por Mbps | Capacidad máxima | Latencia |
|---|---:|---:|---:|
| L1 | $0.08 | 400 Mbps | 20 ms |
| L2 | $0.05 | 500 Mbps | 35 ms |
| L3 | $0.03 | 600 Mbps | 60 ms |

###  Condiciones

- Todo el tráfico debe ser enviado.
- El tráfico puede fraccionarse entre los enlaces.
- No debe superarse la capacidad máxima de cada enlace.
- La latencia promedio ponderada debe ser como máximo **40 ms**.

---

##  Trabajo del estudiante

Formule y resuelva el modelo en PuLP.

Debe determinar:

- variables de decisión;
- tipo de variables;
- función objetivo;
- restricción de balance;
- restricciones de capacidad;
- restricción de latencia promedio;
- costo mínimo;
- distribución óptima de tráfico;
- latencia promedio resultante.


In [5]:
import pulp

# 1. Definición del modelo
problema = pulp.LpProblem("Trafico_WAN", pulp.LpMinimize)

# 2. Variables de decisión continuas con cotas de capacidad
x1 = pulp.LpVariable("L1", lowBound=0, upBound=400, cat="Continuous")
x2 = pulp.LpVariable("L2", lowBound=0, upBound=500, cat="Continuous")
x3 = pulp.LpVariable("L3", lowBound=0, upBound=600, cat="Continuous")

# 3. Función Objetivo
problema += 0.08 * x1 + 0.05 * x2 + 0.03 * x3, "Costo_Total"

# 4. Restricciones
problema += x1 + x2 + x3 == 1000, "Demanda_Total"
problema += 20 * x1 + 35 * x2 + 60 * x3 <= 40000, "Latencia_Promedio_Max"

# 5. Resolver
problema.solve()

# 6. Salidas
print("Estado:", pulp.LpStatus[problema.status])

v1 = x1.varValue if x1.varValue is not None else 0
v2 = x2.varValue if x2.varValue is not None else 0
v3 = x3.varValue if x3.varValue is not None else 0

print(f"L1 = {v1:.2f} Mbps")
print(f"L2 = {v2:.2f} Mbps")
print(f"L3 = {v3:.2f} Mbps")
print(f"Costo = ${pulp.value(problema.objective):.2f}")

# Cálculo limpio de latencia sin caracteres ocultos
latencia = (20 * v1 + 35 * v2 + 60 * v3) / 1000
print(f"Latencia promedio = {latencia:.2f} ms")


Estado: Optimal
L1 = 187.50 Mbps
L2 = 500.00 Mbps
L3 = 312.50 Mbps
Costo = $49.38
Latencia promedio = 40.00 ms



##  Reto de ampliación

1. Reduzca la latencia máxima permitida a **35 ms**.
2. Compare el nuevo costo con el problema original.
3. Luego simule una caída parcial de L2 reduciendo su capacidad a **200 Mbps**.
4. Determine si el problema sigue siendo factible.


In [15]:

import pulp

# Crear el modelo
model = pulp.LpProblem("Enrutamiento_WAN_Reto", pulp.LpMinimize)

# Variables con las capacidades del reto
x1 = pulp.LpVariable("x_L1", lowBound=0, upBound=400, cat=pulp.LpContinuous)
x2 = pulp.LpVariable("x_L2", lowBound=0, upBound=200, cat=pulp.LpContinuous)  # Capacidad reducida a 200
x3 = pulp.LpVariable("x_L3", lowBound=0, upBound=600, cat=pulp.LpContinuous)

# Función objetivo
model += 0.08 * x1 + 0.05 * x2 + 0.03 * x3, "Costo_Total"

# Restricciones
model += x1 + x2 + x3 == 1000, "Balance_Trafico"
model += 20 * x1 + 35 * x2 + 60 * x3 <= 35 * 1000, "Latencia_Maxima_35ms"

# Resolver
status = model.solve()

# Resultado esperado: "Infeasible"
print(f"Estado del modelo: {pulp.LpStatus[status]}")



Estado del modelo: Infeasible



#  Ejercicio 3 — Portafolio de controles de ciberseguridad

##  Planteamiento

El CISO dispone de un presupuesto limitado y debe seleccionar controles de seguridad.  
Cada control tiene un costo y una puntuación estimada de reducción de riesgo.

### 📊 Datos

| Control | Costo | Reducción de riesgo |
|---|---:|---:|
| MFA | 12 | 25 |
| EDR | 20 | 30 |
| SIEM | 25 | 28 |
| PAM | 18 | 24 |
| Backup inmutable | 15 | 22 |
| Capacitación | 8 | 12 |

###  Condiciones

- El presupuesto máximo es **70**.
- SIEM solo puede implementarse si también se selecciona EDR.
- PAM requiere que MFA esté seleccionado.
- Debe elegirse al menos una medida entre **Backup inmutable** y **Capacitación**.
- Deben seleccionarse al menos **4 controles**.

---

## Trabajo del estudiante

Construya un modelo de programación binaria que **maximice la reducción total de riesgo**.

Debe incluir:

- una variable binaria por control;
- función objetivo;
- restricción presupuestaria;
- restricciones de dependencia;
- restricción de continuidad;
- número mínimo de controles;
- interpretación de los controles seleccionados.


In [7]:
import pulp

# 1. Definición del problema
problema = pulp.LpProblem("Ciberseguridad", pulp.LpMaximize)

# 2. Variables binarias
M = pulp.LpVariable("MFA", cat="Binary")
E = pulp.LpVariable("EDR", cat="Binary")
S = pulp.LpVariable("SIEM", cat="Binary")
P = pulp.LpVariable("PAM", cat="Binary")
B = pulp.LpVariable("Backup", cat="Binary")
T = pulp.LpVariable("Capacitacion", cat="Binary")

# 3. Función Objetivo: Maximizar reducción de riesgo
problema += 25 * M + 30 * E + 28 * S + 24 * P + 22 * B + 12 * T, "Reduccion_Total_Riesgo"

# 4. Restricciones
problema += 12 * M + 20 * E + 25 * S + 18 * P + 15 * B + 8 * T <= 70, "Presupuesto_Maximo"
problema += S <= E, "SIEM_requiere_EDR"
problema += P <= M, "PAM_requiere_MFA"
problema += B + T >= 1, "Min_Uno_Backup_o_Capacitacion"
problema += M + E + S + P + B + T >= 4, "Min_4_Controles"

# 5. Resolver
problema.solve()

# 6. Salidas corregidas
print("Estado:", pulp.LpStatus[problema.status])

for variable in problema.variables():
    print(f"{variable.name} = {int(variable.varValue)}")

print(f"Reduccion de riesgo = {pulp.value(problema.objective):.0f}")

costo_usado = sum(c * var.varValue for c, var in zip([25, 12, 20, 18, 15, 8], [S, M, E, P, B, T]))
print(f"Presupuesto consumido = {costo_usado:.0f} / 70")


Estado: Optimal
Backup = 1
Capacitacion = 0
EDR = 1
MFA = 1
PAM = 1
SIEM = 0
Reduccion de riesgo = 101
Presupuesto consumido = 65 / 70



##  Reto de ampliación

Agregue las siguientes reglas:

1. SIEM y una herramienta *legacy* no pueden coexistir.
2. Si se elige **Backup inmutable**, también debe elegirse **MFA**.

Formule las desigualdades binarias correspondientes.

> Nota: si desea calcular un nuevo óptimo incluyendo una herramienta *legacy*, deberá definir también su costo y su contribución a la reducción de riesgo.


In [14]:
import pulp

# 1. Crear el modelo de maximización
modelo_reto = pulp.LpProblem("Reto_Portafolio_Ciberseguridad", pulp.LpMaximize)

# 2. Variables de decisión (Binarias)
x_mfa = pulp.LpVariable("MFA", cat=pulp.LpBinary)
x_edr = pulp.LpVariable("EDR", cat=pulp.LpBinary)
x_siem = pulp.LpVariable("SIEM", cat=pulp.LpBinary)
x_pam = pulp.LpVariable("PAM", cat=pulp.LpBinary)
x_bi = pulp.LpVariable("Backup_Inmutable", cat=pulp.LpBinary)
x_cap = pulp.LpVariable("Capacitacion", cat=pulp.LpBinary)
x_leg = pulp.LpVariable("Legacy", cat=pulp.LpBinary)

# 3. Función Objetivo: Maximizar reducción total de riesgo
modelo_reto += (
    25 * x_mfa +
    30 * x_edr +
    28 * x_siem +
    24 * x_pam +
    22 * x_bi +
    12 * x_cap +
    8 * x_leg
), "Reduccion_Total_Riesgo"

# 4. Restricciones del problema original
modelo_reto += (
    12 * x_mfa +
    20 * x_edr +
    25 * x_siem +
    18 * x_pam +
    15 * x_bi +
    8 * x_cap +
    5 * x_leg <= 70
), "Presupuesto_Maximo"

modelo_reto += x_siem <= x_edr, "SIEM_requiere_EDR"
modelo_reto += x_pam <= x_mfa, "PAM_requiere_MFA"
modelo_reto += x_bi + x_cap >= 1, "Al_menos_uno_Backup_o_Capacitacion"
modelo_reto += x_mfa + x_edr + x_siem + x_pam + x_bi + x_cap + x_leg >= 4, "Minimo_4_Controles"

# 5. Restricciones del reto de ampliación
modelo_reto += x_siem + x_leg <= 1, "Exclusion_SIEM_Legacy"
modelo_reto += x_bi <= x_mfa, "Backup_requiere_MFA"

# 6. Resolver
estado = modelo_reto.solve()

# 7. Salidas con .varValue
print(f"Estado de la solución: {pulp.LpStatus[estado]}")
print(f"Reducción total de riesgo lograda: {pulp.value(modelo_reto.objective):.0f}")

costo_total = (
    12 * x_mfa.varValue +
    20 * x_edr.varValue +
    25 * x_siem.varValue +
    18 * x_pam.varValue +
    15 * x_bi.varValue +
    8 * x_cap.varValue +
    5 * x_leg.varValue
)
print(f"Presupuesto consumido: {costo_total:.0f} / 70\n")

print("Controles seleccionados:")
variables = [x_mfa, x_edr, x_siem, x_pam, x_bi, x_cap, x_leg]
for var in variables:
    estado_str = "SELECCIONADO" if var.varValue == 1 else "NO SELECCIONADO"
    print(f" - {var.name}: {estado_str}")




Estado de la solución: Optimal
Reducción total de riesgo lograda: 109
Presupuesto consumido: 70 / 70

Controles seleccionados:
 - MFA: SELECCIONADO
 - EDR: SELECCIONADO
 - SIEM: NO SELECCIONADO
 - PAM: SELECCIONADO
 - Backup_Inmutable: SELECCIONADO
 - Capacitacion: NO SELECCIONADO
 - Legacy: SELECCIONADO



#  Ejercicio 4 — Distribución de respaldos entre niveles de almacenamiento

##  Planteamiento

Una organización debe almacenar **80 TB** de respaldos utilizando tres niveles: Hot, Warm y Cold.

Se desea **minimizar el costo mensual**, manteniendo una disponibilidad mínima y un tiempo promedio de recuperación aceptable.

###  Datos

| Nivel | Costo por TB | Tiempo de recuperación |
|---|---:|---:|
| Hot | $18 | 0.5 h |
| Warm | $10 | 4 h |
| Cold | $4 | 12 h |

###  Condiciones

- El total almacenado debe ser exactamente **80 TB**.
- Al menos **15 TB** deben permanecer en Hot.
- Al menos **20 TB** deben permanecer en Warm.
- Cold no puede superar **45 TB**.
- El tiempo promedio ponderado de recuperación debe ser como máximo **8 horas**.

---

##  Trabajo del estudiante

Formule y resuelva el modelo en PuLP.

Debe calcular:

- cantidad óptima de TB en cada nivel;
- costo mensual mínimo;
- tiempo promedio de recuperación;
- cumplimiento de todas las restricciones.


In [13]:
import pulp

# 1. Crear el modelo asignándolo a la variable problema
problema = pulp.LpProblem("Almacenamiento", pulp.LpMinimize)

# 2. Variables de decisión continuas
H = pulp.LpVariable("Hot", lowBound=0)
W = pulp.LpVariable("Warm", lowBound=0)
C = pulp.LpVariable("Cold", lowBound=0)

# 3. Función objetivo
problema += 18 * H + 10 * W + 4 * C, "Costo_Total"

# 4. Restricciones
problema += H + W + C == 80, "Capacidad_Total"
problema += H >= 15, "Min_Hot"
problema += W >= 20, "Min_Warm"
problema += C <= 45, "Max_Cold"
problema += 0.5 * H + 4 * W + 12 * C <= 640, "RTO_Maximo"

# 5. Resolver
problema.solve()

# 6. Salidas con .varValue y sin espacios no válidos
print("Estado:", pulp.LpStatus[problema.status])

h_val = H.varValue if H.varValue is not None else 0
w_val = W.varValue if W.varValue is not None else 0
c_val = C.varValue if C.varValue is not None else 0

print(f"Hot = {h_val}")
print(f"Warm = {w_val}")
print(f"Cold = {c_val}")
print(f"Costo = ${pulp.value(problema.objective):.2f}")

rto = (0.5 * h_val + 4 * w_val + 12 * c_val) / 80
print(f"RTO promedio = {rto:.2f} horas")


Estado: Optimal
Hot = 15.0
Warm = 20.0
Cold = 45.0
Costo = $650.00
RTO promedio = 7.84 horas



##  Reto de ampliación

1. Elimine la restricción que limita Cold a **45 TB**.
2. Observe si la restricción de RTO se vuelve determinante.
3. Luego exija un RTO promedio máximo de **6 horas**.
4. Compare la nueva distribución y el costo.


In [12]:
import pulp

# 1. Crear el modelo
problema = pulp.LpProblem("Almacenamiento", pulp.LpMinimize)

# 2. Variables
H = pulp.LpVariable("Hot", lowBound=0)
W = pulp.LpVariable("Warm", lowBound=0)
C = pulp.LpVariable("Cold", lowBound=0)

# 3. Función objetivo
problema += 18*H + 10*W + 4*C, "Costo_Total"

# 4. Restricciones
problema += H + W + C == 80, "Capacidad_Total"
problema += H >= 15, "Min_Hot"
problema += W >= 20, "Min_Warm"

# Verifica el valor 640 contra el enunciado (RTO_max * 80)
problema += 0.5*H + 4*W + 12*C <= 480, "RTO_Maximo"

# 5. Resolver
problema.solve()

# 6. Salidas
print("Estado:", pulp.LpStatus[problema.status])
print("Hot =", H.value())
print("Warm =", W.value())
print("Cold =", C.value())
print("Costo =", pulp.value(problema.objective))

# Cálculo corregido sin espacios no válidos
rto = (0.5 * H.value() + 4 * W.value() + 12 * C.value()) / 80
print("RTO promedio =", rto)


Estado: Optimal
Hot = 15.0
Warm = 38.4375
Cold = 26.5625
Costo = 760.625
RTO promedio = 6.0



#  Ejercicio 5 — Localización de nodos Edge y asignación de regiones

##  Planteamiento

Una compañía debe decidir qué nodos Edge abrir y a qué nodo asignar cada región de usuarios.

Abrir un nodo genera un **costo fijo**.  
Atender una región desde un nodo genera un costo asociado con distancia, latencia y tráfico.

###  Nodos disponibles

| Nodo | Capacidad | Costo fijo |
|---|---:|---:|
| N1 | 80 | 100 |
| N2 | 70 | 90 |
| N3 | 75 | 95 |

### Demandas regionales

| Región | Demanda |
|---|---:|
| R1 | 40 |
| R2 | 35 |
| R3 | 30 |
| R4 | 25 |

### Costos unitarios por región y nodo

| Región | N1 | N2 | N3 |
|---|---:|---:|---:|
| R1 | 2 | 5 | 7 |
| R2 | 4 | 2 | 6 |
| R3 | 6 | 3 | 2 |
| R4 | 7 | 5 | 2 |

###  Condiciones

- Cada región debe asignarse exactamente a **un nodo**.
- Una región solo puede asignarse a un nodo que haya sido abierto.
- La suma de las demandas asignadas a cada nodo no puede superar su capacidad.
- Las decisiones de apertura y asignación son binarias.

---

##  Trabajo del estudiante

Construya un modelo que minimice:

- costos fijos de apertura;
- más costos de servicio de las regiones.

Debe determinar:

- nodos que deben abrirse;
- asignación de cada región;
- costo total;
- utilización de capacidad por nodo.


In [11]:
import pulp

# 1. Definición del modelo
problema = pulp.LpProblem("Localizacion_Nodos_Edge_Normal", pulp.LpMinimize)

# 2. Conjuntos y datos
nodos = ['N1', 'N2', 'N3']
regiones = ['R1', 'R2', 'R3', 'R4']

costo_fijo = {'N1': 100, 'N2': 90, 'N3': 95}
capacidad = {'N1': 80, 'N2': 70, 'N3': 75}
demanda = {'R1': 40, 'R2': 35, 'R3': 30, 'R4': 25}

costo_unitario = {
    'R1': {'N1': 2, 'N2': 5, 'N3': 7},
    'R2': {'N1': 4, 'N2': 2, 'N3': 6},
    'R3': {'N1': 6, 'N2': 3, 'N3': 2},
    'R4': {'N1': 7, 'N2': 5, 'N3': 2}
}

# 3. Variables de decisión binarias
y = pulp.LpVariable.dicts("Abrir_Nodo", nodos, cat=pulp.LpBinary)
x = pulp.LpVariable.dicts("Asignar_Region", [(i, j) for i in regiones for j in nodos], cat=pulp.LpBinary)

# 4. Función Objetivo
costo_apertura = pulp.lpSum(costo_fijo[j] * y[j] for j in nodos)
costo_servicio = pulp.lpSum(costo_unitario[i][j] * demanda[i] * x[(i, j)] for i in regiones for j in nodos)
problema += costo_apertura + costo_servicio, "Costo_Total"

# 5. Restricciones
# Asignación única por región
for i in regiones:
    problema += pulp.lpSum(x[(i, j)] for j in nodos) == 1, f"Asignacion_Unica_{i}"

# Capacidad y enlace con apertura
for j in nodos:
    problema += pulp.lpSum(demanda[i] * x[(i, j)] for i in regiones) <= capacidad[j] * y[j], f"Capacidad_{j}"

# 6. Resolver
problema.solve()

# 7. Salida de resultados
print("Estado del modelo:", pulp.LpStatus[problema.status])
print(f"Costo Total Mínimo: ${pulp.value(problema.objective):.2f}\n")

print("Nodos a abrir y utilización de capacidad:")
for j in nodos:
    if y[j].value() == 1:
        carga_actual = sum(demanda[i] * x[(i, j)].value() for i in regiones)
        porc = (carga_actual / capacidad[j]) * 100
        print(f" - {j}: ABIERTO | Capacidad usada: {carga_actual}/{capacidad[j]} ({porc:.1f}%)")
    else:
        print(f" - {j}: CERRADO")

print("\nAsignación óptima de regiones:")
for i in regiones:
    for j in nodos:
        if x[(i, j)].value() == 1:
            costo_reg = costo_unitario[i][j] * demanda[i]
            print(f" - Región {i} (Demanda: {demanda[i]}) asignada a {j} (Costo: ${costo_reg})")


Estado del modelo: Optimal
Costo Total Mínimo: $525.00

Nodos a abrir y utilización de capacidad:
 - N1: ABIERTO | Capacidad usada: 75.0/80 (93.8%)
 - N2: CERRADO
 - N3: ABIERTO | Capacidad usada: 55.0/75 (73.3%)

Asignación óptima de regiones:
 - Región R1 (Demanda: 40) asignada a N1 (Costo: $80)
 - Región R2 (Demanda: 35) asignada a N1 (Costo: $140)
 - Región R3 (Demanda: 30) asignada a N3 (Costo: $60)
 - Región R4 (Demanda: 25) asignada a N3 (Costo: $50)



##  Reto de ampliación

Analice las siguientes modificaciones:

1. Exigir que se abran al menos **2 nodos**.
2. Exigir que se abran exactamente **2 nodos**.
3. Imponer que **R1 no pueda utilizar N3** debido a un SLA de latencia.

Compare las soluciones obtenidas.


In [10]:
import pulp

# 1. Datos
nodos = ['N1', 'N2', 'N3']
regiones = ['R1', 'R2', 'R3', 'R4']

costo_fijo = {'N1': 100, 'N2': 90, 'N3': 95}
capacidad = {'N1': 80, 'N2': 70, 'N3': 75}
demanda = {'R1': 40, 'R2': 35, 'R3': 30, 'R4': 25}

costo_unitario = {
    'R1': {'N1': 2, 'N2': 5, 'N3': 7},
    'R2': {'N1': 4, 'N2': 2, 'N3': 6},
    'R3': {'N1': 6, 'N2': 3, 'N3': 2},
    'R4': {'N1': 7, 'N2': 5, 'N3': 2}
}

# 2. Modelo
modelo_reto = pulp.LpProblem("Reto_Localizacion_Edge", pulp.LpMinimize)

# Variables de decisión
y = pulp.LpVariable.dicts("Abrir_Nodo", nodos, cat=pulp.LpBinary)
x = pulp.LpVariable.dicts("Asignar_Region", [(i, j) for i in regiones for j in nodos], cat=pulp.LpBinary)

# 3. Función Objetivo
costo_apertura = pulp.lpSum(costo_fijo[j] * y[j] for j in nodos)
costo_servicio = pulp.lpSum(costo_unitario[i][j] * demanda[i] * x[(i, j)] for i in regiones for j in nodos)
modelo_reto += costo_apertura + costo_servicio, "Costo_Total"

# 4. Restricciones Base
for i in regiones:
    modelo_reto += pulp.lpSum(x[(i, j)] for j in nodos) == 1, f"Asignacion_Unica_{i}"

for j in nodos:
    modelo_reto += pulp.lpSum(demanda[i] * x[(i, j)] for i in regiones) <= capacidad[j] * y[j], f"Capacidad_{j}"

# 5. REGLAS DEL RETO DE AMPLIACIÓN
# Regla A: Exactamente 2 nodos abiertos (cumple a su vez con "al menos 2")
modelo_reto += pulp.lpSum(y[j] for j in nodos) == 2, "Exactamente_2_Nodos"

# Regla B: R1 no puede utilizar N3 por SLA
modelo_reto += x[('R1', 'N3')] == 0, "SLA_R1_no_N3"

# 6. Resolver
modelo_reto.solve()

# 7. Resultados
print(f"Estado: {pulp.LpStatus[modelo_reto.status]}")
print(f"Costo Total: ${pulp.value(modelo_reto.objective):.2f}\n")

print("Nodos seleccionados:")
for j in nodos:
    if y[j].value() == 1:
        carga = sum(demanda[i] * x[(i, j)].value() for i in regiones)
        print(f" - {j}: ABIERTO | Carga: {carga}/{capacidad[j]}")
    else:
        print(f" - {j}: CERRADO")

print("\nAsignaciones:")
for i in regiones:
    for j in nodos:
        if x[(i, j)].value() == 1:
            print(f" - {i} -> {j}")



Estado: Optimal
Costo Total: $525.00

Nodos seleccionados:
 - N1: ABIERTO | Carga: 75.0/80
 - N2: CERRADO
 - N3: ABIERTO | Carga: 55.0/75

Asignaciones:
 - R1 -> N1
 - R2 -> N1
 - R3 -> N3
 - R4 -> N3



#  Ejercicio 6 — Dimensionamiento de agentes de CI/CD

## Planteamiento

Una plataforma DevOps necesita capacidad concurrente para pipelines Linux y Windows.

Existen tres tipos de agentes con diferentes capacidades y costos.

###  Datos

| Tipo de agente | Linux slots | Windows slots | Costo |
|---|---:|---:|---:|
| Standard | 4 | 2 | 50 |
| Linux Optimized | 8 | 0 | 70 |
| Universal | 3 | 5 | 80 |

### Condiciones

- Se requieren al menos **40 slots Linux**.
- Se requieren al menos **20 slots Windows**.
- Deben existir al menos **2 agentes Universal**.
- El equipo de operaciones puede administrar como máximo **12 agentes**.

---

##  Trabajo del estudiante

Formule y resuelva un modelo de programación entera que minimice el costo total.

Debe determinar:

- cantidad de agentes Standard;
- cantidad de agentes Linux Optimized;
- cantidad de agentes Universal;
- costo mínimo;
- slots Linux obtenidos;
- slots Windows obtenidos;
- total de agentes utilizados.


In [ ]:
import pulp

# 1. Definir el modelo
problema = pulp.LpProblem("Dimensionamiento_CI_CD", pulp.LpMinimize)

# 2. Variables de decisión (Enteras)
S = pulp.LpVariable("Standard", lowBound=0, cat=pulp.LpInteger)
L = pulp.LpVariable("Linux_Optimized", lowBound=0, cat=pulp.LpInteger)
U = pulp.LpVariable("Universal", lowBound=2, cat=pulp.LpInteger) # u >= 2

# 3. Función Objetivo
problema += 50 * S + 70 * L + 80 * U, "Costo_Total"

# 4. Restricciones
problema += 4 * S + 8 * L + 3 * U >= 40, "Min_Slots_Linux"
problema += 2 * S + 0 * L + 5 * U >= 20, "Min_Slots_Windows"
problema += S + L + U <= 12, "Max_Agentes_Totales"

# 5. Resolver
problema.solve()

# 6. Salidas requeridas
s_val = int(S.value())
l_val = int(L.value())
u_val = int(U.value())
total_agentes = s_val + l_val + u_val
linux_slots = 4 * s_val + 8 * l_val + 3 * u_val
windows_slots = 2 * s_val + 5 * u_val

print(f"Estado: {pulp.LpStatus[problema.status]}")
print(f"Agentes Standard: {s_val}")
print(f"Agentes Linux Optimized: {l_val}")
print(f"Agentes Universal: {u_val}")
print(f"Total de agentes utilizados: {total_agentes} / 12")
print(f"Slots Linux obtenidos: {linux_slots} (mínimo 40)")
print(f"Slots Windows obtenidos: {windows_slots} (mínimo 20)")
print(f"Costo mínimo total: ${pulp.value(problema.objective):.2f}")

Estado: Optimal
Agentes Standard: 5
Agentes Linux Optimized: 2
Agentes Universal: 2
Total de agentes utilizados: 9 / 12
Slots Linux obtenidos: 42 (mínimo 40)
Slots Windows obtenidos: 20 (mínimo 20)
Costo mínimo total: $550.00


## Reto de ampliación

Modifique el modelo de la siguiente manera:

1. Aumente el requerimiento de Windows a **30 slots**.
2. Agregue la regla:

> Por cada 3 agentes Linux Optimized debe existir al menos 1 agente Universal.

Formule matemáticamente dicha restricción e incorpórela al modelo.

Compare el nuevo costo con el problema original.


In [9]:
import pulp

# 1. Crear el modelo del Reto
modelo_reto = pulp.LpProblem("Reto_Dimensionamiento_CI_CD", pulp.LpMinimize)

# 2. Variables enteras
S = pulp.LpVariable("Standard", lowBound=0, cat=pulp.LpInteger)
L = pulp.LpVariable("Linux_Optimized", lowBound=0, cat=pulp.LpInteger)
U = pulp.LpVariable("Universal", lowBound=2, cat=pulp.LpInteger)

# 3. Función Objetivo
modelo_reto += 50 * S + 70 * L + 80 * U, "Costo_Total"

# 4. Restricciones
modelo_reto += 4 * S + 8 * L + 3 * U >= 40, "Min_Slots_Linux"
modelo_reto += 2 * S + 0 * L + 5 * U >= 30, "Min_Slots_Windows_30"  # Aumentado a 30
modelo_reto += S + L + U <= 12, "Max_Agentes_Totales"
modelo_reto += 3 * U >= L, "Regla_Ratio_Universal_Linux"            # Regla de ratio

# 5. Resolver
modelo_reto.solve()

# 6. Mostrar resultados
s_val = int(S.value())
l_val = int(L.value())
u_val = int(U.value())

print("Estado:", pulp.LpStatus[modelo_reto.status])
print(f"Agentes Standard (S): {s_val}")
print(f"Agentes Linux Optimized (L): {l_val}")
print(f"Agentes Universal (U): {u_val}")
print(f"Total de agentes: {s_val + l_val + u_val} / 12")
print(f"Slots Windows obtenidos: {2*s_val + 5*u_val} / 30")
print(f"Slots Linux obtenidos: {4*s_val + 8*l_val + 3*u_val} / 40")
print(f"Costo Mínimo del Reto: ${pulp.value(modelo_reto.objective):.2f}")

Estado: Optimal
Agentes Standard (S): 8
Agentes Linux Optimized (L): 0
Agentes Universal (U): 3
Total de agentes: 11 / 12
Slots Windows obtenidos: 31 / 30
Slots Linux obtenidos: 41 / 40
Costo Mínimo del Reto: $640.00



#  Entrega sugerida

Para cada ejercicio, entregue:

- formulación matemática;
- código en PuLP;
- estado del solver;
- valores de las variables;
- valor de la función objetivo;
- comprobación de restricciones;
- interpretación breve de la solución.

> Si el estado del modelo no es `Optimal`, no interprete los valores de las variables como una solución óptima.
